# ML-05 — Feature Vector and Leakage/Privacy Check

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/jh-emon002/flyrank-intern/blob/main/work/notebooks/w03_feature_leakage_check.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [1]:
%pip -q install duckdb huggingface_hub scikit-learn

In [2]:
import os
import duckdb
import numpy as np
import pandas as pd

try:
    from google.colab import userdata
    HF_TOKEN = userdata.get("HF_TOKEN")
except Exception:
    HF_TOKEN = os.environ.get("HF_TOKEN")

assert HF_TOKEN, "HF_TOKEN not found."

con = duckdb.connect()

con.execute(
    f"""
    CREATE OR REPLACE SECRET hf
    (TYPE huggingface, TOKEN '{HF_TOKEN}')
    """
)

REL = "hf://datasets/FlyRank/internship-warehouse"

MARCH = (
    f"read_parquet("
    f"'{REL}/fact_content_daily_performance/"
    f"month=2026-03/*.parquet'"
    f")"
)

FEATURE_START = "2026-03-01"
FEATURE_END = "2026-03-15"

OUTCOME_START = "2026-03-17"
OUTCOME_END = "2026-03-31"

MIN_IMPRESSIONS = 100
DECLINE_THRESHOLD = 0.80

print("Setup complete.")

Setup complete.


## 1. Build the feature vector

I reuse the data contract from ML-04. One modelling row represents one eligible pseudonymized content item at the March 16 decision point. All honest features are constructed only from March 1–15 observations. The decline proxy is constructed only from March 17–31 observations.

I intentionally keep the feature vector small and interpretable. In addition to the basic search-performance measures, I engineer a logarithmic impression feature, a click-presence flag, and a categorical position band. Identifiers remain context fields and are not model inputs.

In [3]:
page_frame = con.sql(f"""
WITH page_windows AS (

    SELECT
        client_hash_id,
        content_hash_id,

        COUNT(DISTINCT CASE
            WHEN report_date BETWEEN DATE '{FEATURE_START}'
                                 AND DATE '{FEATURE_END}'
             AND gsc_data_available IS TRUE
            THEN report_date
        END) AS feature_days_available,

        COUNT(DISTINCT CASE
            WHEN report_date BETWEEN DATE '{OUTCOME_START}'
                                 AND DATE '{OUTCOME_END}'
             AND gsc_data_available IS TRUE
            THEN report_date
        END) AS outcome_days_available,

        SUM(CASE
            WHEN report_date BETWEEN DATE '{FEATURE_START}'
                                 AND DATE '{FEATURE_END}'
             AND gsc_data_available IS TRUE
            THEN gsc_impressions
            ELSE 0
        END) AS impressions_pre15,

        SUM(CASE
            WHEN report_date BETWEEN DATE '{FEATURE_START}'
                                 AND DATE '{FEATURE_END}'
             AND gsc_data_available IS TRUE
            THEN gsc_clicks
            ELSE 0
        END) AS clicks_pre15,

        SUM(CASE
            WHEN report_date BETWEEN DATE '{FEATURE_START}'
                                 AND DATE '{FEATURE_END}'
             AND gsc_data_available IS TRUE
            THEN gsc_avg_position * gsc_impressions
            ELSE 0
        END)
        /
        NULLIF(
            SUM(CASE
                WHEN report_date BETWEEN DATE '{FEATURE_START}'
                                     AND DATE '{FEATURE_END}'
                 AND gsc_data_available IS TRUE
                THEN gsc_impressions
                ELSE 0
            END),
            0
        ) AS avg_position_pre15,

        STDDEV_POP(CASE
            WHEN report_date BETWEEN DATE '{FEATURE_START}'
                                 AND DATE '{FEATURE_END}'
             AND gsc_data_available IS TRUE
             AND gsc_impressions > 0
            THEN gsc_avg_position
        END) AS position_volatility_pre15,

        COUNT(DISTINCT CASE
            WHEN report_date BETWEEN DATE '{FEATURE_START}'
                                 AND DATE '{FEATURE_END}'
             AND gsc_data_available IS TRUE
             AND gsc_impressions > 0
            THEN report_date
        END) AS days_with_impressions_pre15,

        -- FUTURE OUTCOME: never an honest feature
        SUM(CASE
            WHEN report_date BETWEEN DATE '{OUTCOME_START}'
                                 AND DATE '{OUTCOME_END}'
             AND gsc_data_available IS TRUE
            THEN gsc_impressions
            ELSE 0
        END) AS impressions_next15

    FROM {MARCH}

    GROUP BY
        client_hash_id,
        content_hash_id
)

SELECT *
FROM page_windows

WHERE
    feature_days_available = 15
    AND outcome_days_available = 15
    AND impressions_pre15 >= {MIN_IMPRESSIONS}
""").df()

print("Page-level frame:", page_frame.shape)


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Page-level frame: (58097, 10)


In [4]:
feature_frame = page_frame.copy()

# Honest engineered features
feature_frame["log_impressions_pre15"] = np.log1p(
    feature_frame["impressions_pre15"]
)

feature_frame["ctr_pre15_pct"] = (
    100.0
    * feature_frame["clicks_pre15"]
    / feature_frame["impressions_pre15"]
)

feature_frame["has_clicks_pre15"] = (
    feature_frame["clicks_pre15"] > 0
).astype(int)

# Categorical version of search position
feature_frame["position_band_pre15"] = pd.cut(
    feature_frame["avg_position_pre15"],
    bins=[0, 3, 10, 20, 50, np.inf],
    labels=[
        "top_3",
        "page_1",
        "striking",
        "page_3_5",
        "deep"
    ],
    include_lowest=True
)

# FUTURE / LABEL-DERIVED COLUMNS
feature_frame["decline_ratio"] = (
    feature_frame["impressions_next15"]
    / feature_frame["impressions_pre15"]
)

feature_frame["is_declining_next15d"] = (
    feature_frame["decline_ratio"]
    < DECLINE_THRESHOLD
).astype(int)

NUMERIC_FEATURES = [
    "log_impressions_pre15",
    "ctr_pre15_pct",
    "avg_position_pre15",
    "position_volatility_pre15",
    "days_with_impressions_pre15",
    "has_clicks_pre15",
]

CATEGORICAL_FEATURES = [
    "position_band_pre15",
]

TARGET = "is_declining_next15d"

MODEL_FEATURES = NUMERIC_FEATURES + CATEGORICAL_FEATURES

print("Rows:", len(feature_frame))
print("Honest model features:", MODEL_FEATURES)

feature_frame[MODEL_FEATURES + [TARGET]].head()

Rows: 58097
Honest model features: ['log_impressions_pre15', 'ctr_pre15_pct', 'avg_position_pre15', 'position_volatility_pre15', 'days_with_impressions_pre15', 'has_clicks_pre15', 'position_band_pre15']


,log_impressions_pre15,ctr_pre15_pct,avg_position_pre15,position_volatility_pre15,days_with_impressions_pre15,has_clicks_pre15,position_band_pre15,is_declining_next15d
0,6.063785,0.466200,4.386946,1.739459,15,1,page_1,0
1,6.444131,0.159236,5.265924,2.393685,15,1,page_1,0
2,7.155396,0.703125,4.144531,1.480598,15,1,page_1,0
3,7.836765,0.395101,4.861320,0.902275,15,1,page_1,0
4,7.704361,0.315742,8.977447,1.777931,15,1,page_1,0


## 2. Feature notes (meaning, missing, categorical, available-when?)

Each feature below must pass two tests: I must understand what it represents, and I must have been able to calculate it before the March 16 decision moment. Missing numeric values are not blindly converted to zero; numeric model features will use median imputation with a missingness indicator. Categorical missingness will be represented explicitly through preprocessing. This avoids treating “not measured” as if it necessarily meant a real numerical zero.

In [5]:
feature_notes = pd.DataFrame([
    {
        "feature": "log_impressions_pre15",
        "meaning": "Log-transformed search exposure during Mar 1-15",
        "type": "numeric",
        "missing_handling": "Median + missing indicator",
        "available_when": "Uses only impressions observed before Mar 16"
    },
    {
        "feature": "ctr_pre15_pct",
        "meaning": "Search clicks divided by impressions, ×100",
        "type": "numeric",
        "missing_handling": "Median + missing indicator",
        "available_when": "Clicks and impressions are only from Mar 1-15"
    },
    {
        "feature": "avg_position_pre15",
        "meaning": "Impression-weighted average search position",
        "type": "numeric",
        "missing_handling": "Median + missing indicator",
        "available_when": "Position observations are only from Mar 1-15"
    },
    {
        "feature": "position_volatility_pre15",
        "meaning": "Variation in daily search position",
        "type": "numeric",
        "missing_handling": "Median + missing indicator",
        "available_when": "All position values occur before Mar 16"
    },
    {
        "feature": "days_with_impressions_pre15",
        "meaning": "Number of feature-window days with search visibility",
        "type": "numeric",
        "missing_handling": "Median + missing indicator",
        "available_when": "Counts only Mar 1-15 days"
    },
    {
        "feature": "has_clicks_pre15",
        "meaning": "Whether the page received at least one search click",
        "type": "binary",
        "missing_handling": "No fill expected",
        "available_when": "Uses clicks observed only before Mar 16"
    },
    {
        "feature": "position_band_pre15",
        "meaning": "Categorical band derived from average search position",
        "type": "categorical",
        "missing_handling": "Most frequent / unknown-safe encoding",
        "available_when": "Derived only from pre-decision position"
    }
])

feature_notes


,feature,meaning,type,missing_handling,available_when
0,log_impressions_pre15,Log-transformed search exposure during Mar 1-15,numeric,Median + missing indicator,Uses only impressions observed before Mar 16
1,ctr_pre15_pct,"Search clicks divided by impressions, ×100",numeric,Median + missing indicator,Clicks and impressions are only from Mar 1-15
2,avg_position_pre15,Impression-weighted average search position,numeric,Median + missing indicator,Position observations are only from Mar 1-15
3,position_volatility_pre15,Variation in daily search position,numeric,Median + missing indicator,All position values occur before Mar 16
4,days_with_impressions_pre15,Number of feature-window days with search visi...,numeric,Median + missing indicator,Counts only Mar 1-15 days
5,has_clicks_pre15,Whether the page received at least one search ...,binary,No fill expected,Uses clicks observed only before Mar 16
6,position_band_pre15,Categorical band derived from average search p...,categorical,Most frequent / unknown-safe encoding,Derived only from pre-decision position


In [6]:
missing_check = pd.DataFrame({
    "missing_count":
        feature_frame[MODEL_FEATURES].isna().sum(),

    "missing_pct":
        feature_frame[MODEL_FEATURES].isna().mean() * 100
})

missing_check.round(2)

,missing_count,missing_pct
log_impressions_pre15,0,0.0
ctr_pre15_pct,0,0.0
avg_position_pre15,0,0.0
position_volatility_pre15,0,0.0
days_with_impressions_pre15,0,0.0
has_clicks_pre15,0,0.0
position_band_pre15,0,0.0


In [7]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline

numeric_pipe = Pipeline([
    (
        "imputer",
        SimpleImputer(
            strategy="median",
            add_indicator=True
        )
    )
])

categorical_pipe = Pipeline([
    (
        "imputer",
        SimpleImputer(
            strategy="most_frequent"
        )
    ),
    (
        "onehot",
        OneHotEncoder(
            handle_unknown="ignore"
        )
    )
])

preprocessor = ColumnTransformer([
    (
        "numeric",
        numeric_pipe,
        NUMERIC_FEATURES
    ),
    (
        "categorical",
        categorical_pipe,
        CATEGORICAL_FEATURES
    )
])

print("Preprocessing pipeline created.")

Preprocessing pipeline created.


## 3. The leakage hunt

I attack the feature vector before trusting its score. I check for label-derived fields, future-window information, existing decision/product flags, context IDs, and potentially identifying text fields. I then deliberately add one known leaky feature, compare its validation score with the honest feature set, and remove it again.

In [8]:
LEAKAGE_TERMS = [
    "label",
    "target",
    "future",
    "next",
    "outcome",
    "decline_ratio",
    "trend",
    "recommend",
    "opportunity",
    "score"
]

suspect_model_features = [
    col for col in MODEL_FEATURES
    if any(term in col.lower()
           for term in LEAKAGE_TERMS)
]

print(
    "Suspicious names inside honest features:",
    suspect_model_features
)


Suspicious names inside honest features: []


In [9]:
ID_COLUMNS = {
    "client_hash_id",
    "content_hash_id"
}

id_leaks = ID_COLUMNS.intersection(
    MODEL_FEATURES
)

print("IDs used as model features:", id_leaks)

assert len(id_leaks) == 0

IDs used as model features: set()


In [10]:
PRIVATE_NAME_PATTERNS = [
    "url",
    "domain",
    "client_name",
    "query_text",
    "keyword_text",
    "title"
]

privacy_hits = [
    col for col in feature_frame.columns
    if any(term in col.lower()
           for term in PRIVATE_NAME_PATTERNS)
]

print("Potentially identifying columns found:", privacy_hits)

Potentially identifying columns found: []


In [11]:
from sklearn.model_selection import GroupShuffleSplit
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score

model_df = feature_frame.dropna(
    subset=[TARGET]
).copy()

X = model_df[MODEL_FEATURES]
y = model_df[TARGET]

groups = model_df["client_hash_id"]

splitter = GroupShuffleSplit(
    n_splits=1,
    test_size=0.25,
    random_state=42
)

train_idx, test_idx = next(
    splitter.split(X, y, groups=groups)
)

X_train = X.iloc[train_idx]
X_test = X.iloc[test_idx]

y_train = y.iloc[train_idx]
y_test = y.iloc[test_idx]

In [12]:
from sklearn.model_selection import GroupShuffleSplit
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score

model_df = feature_frame.dropna(
    subset=[TARGET]
).copy()

X = model_df[MODEL_FEATURES]
y = model_df[TARGET]

groups = model_df["client_hash_id"]

splitter = GroupShuffleSplit(
    n_splits=1,
    test_size=0.25,
    random_state=42
)

train_idx, test_idx = next(
    splitter.split(X, y, groups=groups)
)

X_train = X.iloc[train_idx]
X_test = X.iloc[test_idx]

y_train = y.iloc[train_idx]
y_test = y.iloc[test_idx]

In [13]:
test_positive_rate = y_test.mean()

majority_baseline = max(
    test_positive_rate,
    1 - test_positive_rate
)

print(
    "Test positive rate:",
    round(test_positive_rate, 3)
)

print(
    "Majority-class accuracy baseline:",
    round(majority_baseline, 3)
)

Test positive rate: 0.36
Majority-class accuracy baseline: 0.64


In [14]:
honest_pipeline = Pipeline([
    ("prep", preprocessor),
    (
        "model",
        DecisionTreeClassifier(
            max_depth=4,
            min_samples_leaf=20,
            random_state=42
        )
    )
])

honest_pipeline.fit(
    X_train,
    y_train
)

honest_pred = honest_pipeline.predict(
    X_test
)

honest_accuracy = accuracy_score(
    y_test,
    honest_pred
)

print(
    "Honest grouped accuracy:",
    round(honest_accuracy, 3)
)

Honest grouped accuracy: 0.654


In [15]:
LEAKY_NUMERIC_FEATURES = (
    NUMERIC_FEATURES
    + ["decline_ratio"]
)

leaky_preprocessor = ColumnTransformer([
    (
        "numeric",
        Pipeline([
            (
                "imputer",
                SimpleImputer(
                    strategy="median",
                    add_indicator=True
                )
            )
        ]),
        LEAKY_NUMERIC_FEATURES
    ),
    (
        "categorical",
        categorical_pipe,
        CATEGORICAL_FEATURES
    )
])

LEAKY_FEATURES = (
    LEAKY_NUMERIC_FEATURES
    + CATEGORICAL_FEATURES
)

X_leaky = model_df[LEAKY_FEATURES]

X_train_leaky = X_leaky.iloc[train_idx]
X_test_leaky = X_leaky.iloc[test_idx]

leaky_pipeline = Pipeline([
    ("prep", leaky_preprocessor),
    (
        "model",
        DecisionTreeClassifier(
            max_depth=4,
            min_samples_leaf=20,
            random_state=42
        )
    )
])

leaky_pipeline.fit(
    X_train_leaky,
    y_train
)

leaky_pred = leaky_pipeline.predict(
    X_test_leaky
)

leaky_accuracy = accuracy_score(
    y_test,
    leaky_pred
)

print(
    "Honest grouped accuracy:",
    round(honest_accuracy, 3)
)

print(
    "Leaky grouped accuracy:",
    round(leaky_accuracy, 3)
)

print(
    "Score increase:",
    round(
        leaky_accuracy - honest_accuracy,
        3
    )
)

Honest grouped accuracy: 0.654
Leaky grouped accuracy: 1.0
Score increase: 0.346


In [16]:
FINAL_FEATURES = MODEL_FEATURES.copy()

assert "decline_ratio" not in FINAL_FEATURES
assert "impressions_next15" not in FINAL_FEATURES
assert TARGET not in FINAL_FEATURES

print("FINAL HONEST FEATURE VECTOR")

for feature in FINAL_FEATURES:
    print("-", feature)

print(
    "\nRetained honest accuracy:",
    round(honest_accuracy, 3)
)

FINAL HONEST FEATURE VECTOR
- log_impressions_pre15
- ctr_pre15_pct
- avg_position_pre15
- position_volatility_pre15
- days_with_impressions_pre15
- has_clicks_pre15
- position_band_pre15

Retained honest accuracy: 0.654


The leaky model received `decline_ratio`, which is directly calculated from the future-window impressions used to define `is_declining_next15d`. Its improved score therefore does not demonstrate predictive skill. It demonstrates that the model was given information derived from the answer. I removed the column and retain the grouped honest score.

## 4. What I excluded and why

The final feature vector contains only pre-decision aggregate search signals. The following fields are deliberately excluded either because they describe the outcome, encode the label, identify/group entities rather than describe them, represent operational/product decisions, or do not satisfy the privacy/time contract.

In [17]:
excluded_fields = pd.DataFrame([
    {
        "field": "impressions_next15",
        "reason": "Future outcome-window information; unavailable on Mar 16"
    },
    {
        "field": "decline_ratio",
        "reason": "Derived from future impressions and directly determines the label"
    },
    {
        "field": "is_declining_next15d",
        "reason": "This is the prediction target itself"
    },
    {
        "field": "client_hash_id",
        "reason": "Context/grouping field; used for client-holdout validation, not learning"
    },
    {
        "field": "content_hash_id",
        "reason": "Context/join key; not a model feature"
    },
    {
        "field": "report_date",
        "reason": "Used to enforce feature/outcome windows, not used as a learned signal"
    },
    {
        "field": "gsc_data_available",
        "reason": "Used as an availability filter rather than a predictive signal"
    },
    {
        "field": "fact_content_query_90d recent-window fields",
        "reason": "Snapshot-relative window is not aligned with the Mar 16 prediction moment"
    },
    {
        "field": "existing recommendation/opportunity scores",
        "reason": "Decision/product-derived signals would teach the model the old rule"
    },
    {
        "field": "URLs/domains/raw titles/raw queries",
        "reason": "Not needed and excluded for privacy/data-use reasons"
    }
])

excluded_fields


,field,reason
0,impressions_next15,Future outcome-window information; unavailable...
1,decline_ratio,Derived from future impressions and directly d...
2,is_declining_next15d,This is the prediction target itself
3,client_hash_id,Context/grouping field; used for client-holdou...
4,content_hash_id,Context/join key; not a model feature
5,report_date,"Used to enforce feature/outcome windows, not u..."
6,gsc_data_available,Used as an availability filter rather than a p...
7,fact_content_query_90d recent-window fields,Snapshot-relative window is not aligned with t...
8,existing recommendation/opportunity scores,Decision/product-derived signals would teach t...
9,URLs/domains/raw titles/raw queries,Not needed and excluded for privacy/data-use r...


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.